In [ ]:
# ===== Push analysis outputs to GitHub (REST API; no Git install needed) =====
import os, glob, base64, getpass, requests
from urllib.parse import quote

OWNER, REPO, BRANCH = 'dac0009', 'Forest-Management-Carbon', 'main'
DEST = 'analysis'                                   # folder created in the repo
NB_DIR = r'C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\Fernow_Notebooks'
PROJECT_DIR = r'C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026'
FIA_OUT = r'C:\Users\DCifelli2\Downloads\WV_CSV_filtered'   # only if you've run notebook 00

files = (glob.glob(os.path.join(NB_DIR, '*.ipynb'))
         + [os.path.join(NB_DIR, f) for f in ['fernow_stats.py', 'README.md',
                                              'soil_mechanism_tests.csv', 'fernow_census_periods.csv']]
         + glob.glob(os.path.join(PROJECT_DIR, 'New Notebook*.ipynb'))      # the notebooks you actually ran in
         + [os.path.join(FIA_OUT, f) for f in ['fia_cond_live.csv.gz', 'fia_cond_seedlings.csv.gz', 'MANIFEST.csv']])
files = [f for f in files if os.path.exists(f)]
print('Will upload:'); [print('  ', os.path.basename(f), f'{os.path.getsize(f)/1e6:.1f} MB') for f in files]

token = getpass.getpass('Paste GitHub token (hidden): ')
H = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'}
api = f'https://api.github.com/repos/{OWNER}/{REPO}/contents/'

for f in files:
    mb = os.path.getsize(f) / 1e6
    if mb > 50:
        print(f'SKIP {os.path.basename(f)} ({mb:.0f} MB) — too large for this method'); continue
    sub = 'fia/' if f.startswith(FIA_OUT) else ('project_notebooks/' if f.startswith(PROJECT_DIR) else '')
    rel = f'{DEST}/{sub}{os.path.basename(f)}'
    url = api + quote(rel)
    r = requests.get(url, headers=H, params={'ref': BRANCH})
    body = {'message': f'Add/update {rel}', 'branch': BRANCH,
            'content': base64.b64encode(open(f, 'rb').read()).decode()}
    if r.status_code == 200:
        body['sha'] = r.json()['sha']               # file exists -> update it
    r = requests.put(url, headers=H, json=body)
    print('OK  ' if r.ok else f'FAIL {r.status_code}', rel, '' if r.ok else r.json().get('message', ''))
del token

Will upload:
   00_FIA_Filter_Local.ipynb 0.0 MB
   01_Fernow_Core_Stats.ipynb 0.0 MB
   02_Fernow_History_Terrain.ipynb 0.0 MB
   03_Soil_Mechanisms.ipynb 0.0 MB
   04_FIA_WV_Context.ipynb 0.0 MB
   fernow_stats.py 0.0 MB
   README.md 0.0 MB
   soil_mechanism_tests.csv 0.0 MB
   New Notebook (1).ipynb 0.1 MB
   New Notebook (2).ipynb 0.2 MB
   New Notebook (3).ipynb 0.4 MB
   New Notebook (4).ipynb 0.0 MB
   New Notebook.ipynb 0.5 MB
   fia_cond_live.csv.gz 0.6 MB
   fia_cond_seedlings.csv.gz 0.2 MB
   MANIFEST.csv 0.0 MB


Paste GitHub token (hidden):  ········
